# 4-1절 연습 문제 풀이

이 노트북은 4-1절 연습 문제(4-1 ~ 4-3)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch04/04-01_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import copy
import csv
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = '../../data'
LR = 0.01
EPOCHS = 10000
PATIENCE = 100
HIDDEN_DIM = 64

def load_spiral():
    with open(f'{DATA_DIR}/ch3_spiral_data.csv', 'r') as f:
        rows = list(csv.DictReader(f))
    X = torch.tensor([[float(r['x1']), float(r['x2'])] for r in rows])
    Y = torch.tensor([int(r['label']) for r in rows])
    return X, Y

def split_three(X, Y, ratios=(0.6, 0.2, 0.2), seed=SEED):
    """훈련, 검증, 평가 데이터셋으로 나눈다."""
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(X), generator=generator)
    X, Y = X[order], Y[order]
    n_train = int(len(X) * ratios[0])
    n_valid = int(len(X) * ratios[1])
    return (X[:n_train], Y[:n_train],
            X[n_train:n_train + n_valid], Y[n_train:n_train + n_valid],
            X[n_train + n_valid:], Y[n_train + n_valid:])

def build_mlp(activation=nn.ReLU, out_features=3, hidden_dim=HIDDEN_DIM, last_activation=None):
    layers = [nn.Linear(2, hidden_dim), activation(),
              nn.Linear(hidden_dim, hidden_dim), activation(),
              nn.Linear(hidden_dim, out_features)]
    if last_activation is not None:
        layers.append(last_activation())
    return nn.Sequential(*layers)

X_all, Y_all = load_spiral()
X_train, Y_train, X_valid, Y_valid, X_test, Y_test = split_three(X_all, Y_all)
print(f'훈련 {len(X_train)}개, 검증 {len(X_valid)}개, 평가 {len(X_test)}개')

훈련 540개, 검증 180개, 평가 180개


In [2]:
def train_with_early_stopping(model, criterion, optimizer, epochs=EPOCHS, patience=PATIENCE,
                              X_train=None, Y_train=None, X_valid=None, Y_valid=None,
                              monitor='loss'):
    """[코드 4-3]과 같은 조기 종료 학습 함수. monitor로 감시 지표를 고른다."""
    best_score = float('inf') if monitor == 'loss' else -float('inf')
    best_epoch, best_params, counter = 0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train), Y_train)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_out = model(X_valid)
            valid_loss = criterion(valid_out, Y_valid).item()
            # 정답이 원-핫 텐서면 인덱스로 되돌려 비교한다
            Y_index = Y_valid.argmax(dim=-1) if Y_valid.dim() > 1 else Y_valid
            valid_acc = (valid_out.argmax(dim=-1) == Y_index).float().mean().item()
        score = valid_loss if monitor == 'loss' else valid_acc
        improved = score < best_score if monitor == 'loss' else score > best_score
        if improved:
            best_score, best_epoch, counter = score, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                break
    model.load_state_dict(best_params)
    return best_epoch, best_score, epoch

def accuracy(model, X, Y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=-1) == Y).float().mean().item() * 100

## 연습 문제 4-1

> 조기 종료 방식으로 모델을 학습하는 [코드 4-3] 예제에서 참을성 한계를 늘리거나 줄이면 모델의 학습 과정이
> 어떻게 바뀔지 예측한 후, 직접 값을 바꿔 가며 실제 결과가 예측과 일치하는지 확인해 보자.

### 예측

본문 p9의 설명대로라면 이렇게 예상할 수 있다.

- **참을성 한계를 줄이면**: 일시적인 정체를 과적합으로 오판해 일찍 멈춘다. 종료 에포크가 앞당겨지고,
  덜 학습한 상태에서 끝나므로 성능이 떨어질 수 있다.
- **참을성 한계를 늘리면**: 오판은 줄지만 과적합 구간을 더 학습하므로 시간이 낭비된다.
  다만 **복원하는 모델은 검증 손실이 최소였던 시점의 것**이므로, 한계를 늘려도 최종 성능은 크게 달라지지 않는다.

마지막 항목이 이 문제의 핵심이다. 참을성 한계는 **언제 멈출지**를 정할 뿐 **어떤 모델을 남길지**는 정하지 않는다.

In [3]:
PATIENCES = [10, 30, 100, 300, 1000]

print(f'{"참을성":>6} {"최적 에포크":>10} {"종료 에포크":>10} {"최소 검증 손실":>13} {"평가 정확도":>11}')
print('-' * 60)
for patience in PATIENCES:
    torch.manual_seed(SEED)
    model = build_mlp()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    best_epoch, best_loss, last_epoch = train_with_early_stopping(
        model, criterion, optimizer, patience=patience,
        X_train=X_train, Y_train=Y_train, X_valid=X_valid, Y_valid=Y_valid)
    print(f'{patience:6d} {best_epoch:10d} {last_epoch:10d} {best_loss:13.4f} '
          f'{accuracy(model, X_test, Y_test):10.2f}%')

   참을성     최적 에포크     종료 에포크      최소 검증 손실      평가 정확도
------------------------------------------------------------


    10        195        205        0.1207      92.78%


    30        245        275        0.1153      92.78%


   100        245        345        0.1153      92.78%


   300        449        749        0.1132      93.33%


  1000        449       1449        0.1132      93.33%


### 풀이 해설

결과를 보면 예측이 맞는 부분과 빗나가는 부분이 함께 드러난다.

| 참을성 | 최적 에포크 | 종료 에포크 | 최소 검증 손실 | 평가 정확도 |
|---|---|---|---|---|
| 10 | 195 | 205 | 0.1207 | 92.78% |
| 30 | 245 | 275 | 0.1153 | 92.78% |
| 100 | 245 | 345 | 0.1153 | 92.78% |
| 300 | 449 | 749 | 0.1132 | 93.33% |
| 1000 | 449 | 1449 | 0.1132 | 93.33% |

**참을성 한계가 작으면 최적 에포크 자체가 앞쪽에 머문다.** 한계가 10일 때는 195 에포크에서 멈췄는데,
이는 그 뒤의 정체 구간을 넘기지 못해 **더 좋은 지점이 있었는데도 도달하지 못했다**는 뜻이다.
한계를 300으로 키우자 449 에포크라는 더 나은 지점을 찾아냈고 평가 정확도도 92.78%에서 93.33%로 올랐다.

**반면 어느 지점을 넘어서면 한계를 더 키워도 결과가 같아진다.** 300과 1,000은 최적 에포크(449)도
검증 손실(0.1132)도 평가 정확도(93.33%)도 완전히 같다. 달라지는 것은 종료 에포크, 곧 **학습에 들인 시간**뿐이다.
한계가 1,000일 때는 449 에포크 이후 1,000 에포크를 더 돌고 나서야 멈췄지만, 복원되는 모델은 같다.

정리하면 **참을성 한계를 키우는 비용은 시간이고, 줄이는 비용은 성능이다.** 그리고 어느 선을 넘으면
시간만 더 들 뿐 성능은 나아지지 않는다. 본문 p9가 "학습 상황과 비용을 함께 고려해 결정해야 한다"고 한
이유가 이 표에 그대로 나타난다.

### 문제 검토

- **적절성: 적합. 4-1절을 마무리하는 좋은 문제다.** '예측한 후 확인하라'는 구성이 특히 좋다.
  독자가 본문의 설명을 자기 말로 정리한 뒤 실험으로 검증하게 만든다. 4장 연습 문제 중 가장 잘 설계됐다.
- **[검토] 무엇을 비교할지 짚어 주면 좋다.** '학습 과정이 어떻게 바뀔지'만으로는 종료 에포크만 보고 끝낼 수 있다.
  이 문제에서 가장 중요한 관찰은 **참을성 한계를 키워도 복원되는 모델은 같다**는 점인데, 평가 정확도를 함께
  보지 않으면 놓친다.

**윤문안**

> **4-1**. 조기 종료 방식으로 모델을 학습하는 [코드 4-3] 예제에서 참을성 한계를 늘리거나 줄이면 모델의 학습 과정이
> 어떻게 바뀔지 예측한 후, 직접 값을 바꿔 가며 실제 결과가 예측과 일치하는지 확인해 보자.
> 이때 학습이 끝난 에포크뿐 아니라 검증 손실이 최소였던 에포크와 평가 정확도도 함께 비교해 보자.

## 연습 문제 4-2

> 3-3절의 회오리 모양 데이터 분류 모델 중 평균제곱오차를 사용해 학습한 모델과, 교차 엔트로피 손실 함수에
> 은닉층 시그모이드 활성화를 사용한 모델 각각에 조기 종료 방식을 적용해 학습해 보자.
> 최적의 모델을 찾아 정확도를 확인하고 [코드 4-3]의 결과와 비교해 보자.
>
> 힌트: 두 모델과 [코드 4-3]은 모두 검증 손실이 정체되는 형태가 다를 수 있다. 최적의 모델을 찾으려면
> 단순히 조기 종료 방식만 적용해서는 안 되고 다른 하이퍼파라미터의 수정이 필요할 수도 있다.

In [4]:
import torch.nn.functional as F

Y_train_onehot = F.one_hot(Y_train, num_classes=3).float()
Y_valid_onehot = F.one_hot(Y_valid, num_classes=3).float()

def run_case(name, activation, last_activation, criterion, Y_tr, Y_va, patience=PATIENCE, lr=LR):
    torch.manual_seed(SEED)
    model = build_mlp(activation=activation, last_activation=last_activation)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_epoch, best_loss, last_epoch = train_with_early_stopping(
        model, criterion, optimizer, patience=patience,
        X_train=X_train, Y_train=Y_tr, X_valid=X_valid, Y_valid=Y_va)
    acc = accuracy(model, X_test, Y_test)
    print(f'{name:28} 최적 {best_epoch:5d} / 종료 {last_epoch:5d} 에포크, '
          f'최소 검증 손실 {best_loss:.4f}, 평가 정확도 {acc:.2f}%')
    return acc

print('[본문과 같은 참을성 한계 100]')
run_case('① MSE + 시그모이드', nn.Sigmoid, nn.Sigmoid, nn.MSELoss(), Y_train_onehot, Y_valid_onehot)
run_case('② 교차 엔트로피 + 시그모이드', nn.Sigmoid, None, nn.CrossEntropyLoss(), Y_train, Y_valid)
run_case('③ 교차 엔트로피 + ReLU', nn.ReLU, None, nn.CrossEntropyLoss(), Y_train, Y_valid)

[본문과 같은 참을성 한계 100]


① MSE + 시그모이드                최적   943 / 종료  1043 에포크, 최소 검증 손실 0.0262, 평가 정확도 93.33%


② 교차 엔트로피 + 시그모이드            최적   567 / 종료   667 에포크, 최소 검증 손실 0.1350, 평가 정확도 92.78%


③ 교차 엔트로피 + ReLU             최적   245 / 종료   345 에포크, 최소 검증 손실 0.1153, 평가 정확도 92.78%


92.77777671813965

In [5]:
print('[참을성 한계를 늘리고 학습률을 조정한 경우]')
run_case('① MSE + 시그모이드', nn.Sigmoid, nn.Sigmoid, nn.MSELoss(),
         Y_train_onehot, Y_valid_onehot, patience=2000, lr=0.05)
run_case('② 교차 엔트로피 + 시그모이드', nn.Sigmoid, None, nn.CrossEntropyLoss(),
         Y_train, Y_valid, patience=2000, lr=0.05)

[참을성 한계를 늘리고 학습률을 조정한 경우]


① MSE + 시그모이드                최적   323 / 종료  2323 에포크, 최소 검증 손실 0.0303, 평가 정확도 92.22%


② 교차 엔트로피 + 시그모이드            최적   845 / 종료  2845 에포크, 최소 검증 손실 0.1259, 평가 정확도 94.44%


94.44444179534912

### 풀이 해설

본문과 같은 참을성 한계 100으로 세 모델을 학습한 결과다.

| 모델 | 최적 에포크 | 종료 에포크 | 평가 정확도 |
|---|---|---|---|
| ① 평균제곱오차 + 시그모이드 | 943 | 1,043 | 93.33% |
| ② 교차 엔트로피 + 시그모이드 | 567 | 667 | 92.78% |
| ③ 교차 엔트로피 + ReLU | 245 | 345 | 92.78% |

**ReLU 모델이 가장 빨리 최적 시점에 도달한다**(245 에포크). 3장에서 본 "ReLU는 기울기 소실이 없어
학습이 빠르다"는 결론이 조기 종료에서도 그대로 확인된다. 시그모이드 모델은 각각 567, 943 에포크가 걸렸다.

그런데 **정확도는 세 모델이 92~93%로 거의 같다.** 3장에서는 평균제곱오차 모델 91.94%, ReLU 모델 93.33%로
차이가 있었는데, 조기 종료를 적용하자 그 격차가 사라졌다. 3장의 차이가 모델의 우열이라기보다
**과적합 구간까지 학습한 결과였다**는 뜻이다. 조기 종료가 뒤처진 모델을 끌어올린 셈이다.
이것이 이 문제에서 얻을 수 있는 가장 값진 관찰이다.

힌트가 말하는 하이퍼파라미터 조정도 확인해 보면, 참을성 한계를 2,000으로 늘리고 학습률을 0.05로 올렸을 때
② 모델은 94.44%까지 올랐지만 ① 모델은 92.22%로 오히려 떨어졌다.
**조정이 항상 개선으로 이어지지는 않는다**는 점도 함께 확인할 수 있다.

### 문제 검토

- **적절성: 적합. 3장과 4장을 잇는 좋은 문제다.** 3장에서 만든 세 모델에 4장의 조기 종료를 적용해,
  '손실 함수와 활성화 함수의 선택이 학습 속도를 바꾸고, 학습 속도가 다시 조기 종료 설정을 바꾼다'는
  연쇄를 체감하게 한다. 힌트도 정확히 그 방향을 가리킨다.
- **[검토] '최적의 모델을 찾아'가 모호하다.** 무엇을 기준으로 최적인지(검증 손실인지 평가 정확도인지),
  어떤 하이퍼파라미터를 얼마나 바꿔 볼지가 열려 있어 독자마다 결과가 크게 달라진다.
  도전 문제로 분류하거나 탐색 범위를 좁혀 주는 편이 좋다.
- **[검토] 세 모델의 정의를 다시 찾아야 한다.** 3-3절의 세 모델이 무엇이었는지 기억나지 않으면 되돌아가야 한다.
  "[코드 3-18], [코드 3-25], [코드 3-28]의 세 모델"처럼 코드 번호로 짚어 주면 친절하다.

## 연습 문제 4-3 [도전 문제]

> [코드 4-3] 예제에서는 검증 손실을 조기 종료 여부를 판단하는 지표로 사용한다. 그런데 분류 모델에서는
> 손실만큼이나 분류 정확도도 중요한 검증 지표로 활용할 수 있다.
> 분류 정확도를 검증 지표로 사용하도록 [코드 4-3] 예제를 수정한 후, 결과를 확인해 보자.

In [6]:
# 위 학습 함수의 monitor 인자로 감시 지표를 바꾼다
# 손실은 작을수록, 정확도는 클수록 좋으므로 개선 여부를 판정하는 부등호 방향이 반대다
for monitor, label in (('loss', '검증 손실 기준'), ('accuracy', '검증 정확도 기준')):
    torch.manual_seed(SEED)
    model = build_mlp()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    best_epoch, best_score, last_epoch = train_with_early_stopping(
        model, criterion, optimizer, monitor=monitor,
        X_train=X_train, Y_train=Y_train, X_valid=X_valid, Y_valid=Y_valid)
    score_text = f'{best_score:.4f}' if monitor == 'loss' else f'{best_score * 100:.2f}%'
    print(f'{label:14} 최적 {best_epoch:5d} / 종료 {last_epoch:5d} 에포크, '
          f'최적 지표 {score_text}, 평가 정확도 {accuracy(model, X_test, Y_test):.2f}%')

검증 손실 기준       최적   245 / 종료   345 에포크, 최적 지표 0.1153, 평가 정확도 92.78%


검증 정확도 기준      최적   131 / 종료   231 에포크, 최적 지표 96.11%, 평가 정확도 92.22%


### 풀이 해설

구현에서 바꿔야 할 곳은 **세 군데**다.

1. 감시할 값을 검증 손실에서 검증 정확도로 바꾼다.
2. 최초값을 `inf` 대신 `-inf`로 둔다.
3. **개선 판정의 부등호 방향을 뒤집는다.** 손실은 작아져야 개선이지만 정확도는 커져야 개선이다.

세 번째가 가장 실수하기 쉬운 지점이다. 부등호를 그대로 두면 학습이 거의 즉시 멈춘다.

결과를 보면 **정확도를 기준으로 삼은 쪽이 훨씬 이른 131 에포크에서 멈춘다**(손실 기준은 245 에포크).
평가 정확도도 92.22%로 손실 기준(92.78%)보다 조금 낮다.

이유는 두 지표의 성질 차이에 있다. 검증 정확도는 예측이 문턱값을 넘는지만 보므로 **계단처럼 띄엄띄엄 변하고
같은 값이 자주 반복된다.** 180개 샘플로 계산하면 정확도가 가질 수 있는 값은 181가지뿐이다.
한 번 96.11%를 찍고 나면 그보다 나아지기 어려워 갱신이 멈추고, 참을성 카운터가 금세 한계에 도달한다.
반면 검증 손실은 연속값이라 미세한 개선도 갱신으로 잡아낸다.

즉 **정확도는 사람이 이해하기 쉽지만 조기 종료의 감시 지표로는 둔감하다.**
실무에서 검증 손실을 기본 감시 지표로 삼는 이유가 여기에 있다.
다만 손실이 나빠지는데도 정확도가 유지되는 경우처럼 두 지표가 어긋나기도 하므로, 함께 보는 것이 좋다.

### 문제 검토

- **적절성: 적합. 도전 문제로 알맞다.** 지표를 바꾸는 단순한 수정처럼 보이지만 부등호 방향이라는 함정이 있어
  코드를 이해하지 못하면 통과할 수 없다. 또 '손실과 정확도가 어떻게 다른 지표인가'라는 질문으로 이어져
  4-3절의 분류 정확도 논의와도 맞닿는다.
- **[검토] 무엇을 비교해야 하는지 없다.** '결과를 확인해 보자'만으로는 두 방식의 차이를 놓치기 쉽다.
  최적 에포크와 평가 정확도를 함께 비교하라고 하면 '정확도는 계단처럼 변해 둔감한 지표'라는 관찰에 도달한다.
- **[검토] 부등호 함정을 힌트로 남길지 판단이 필요하다.** 도전 문제이므로 그대로 두어 스스로 발견하게 하는 것도
  좋고, "개선 여부를 판단하는 조건이 손실과 정확도에서 어떻게 달라지는지 주의하자" 정도로 살짝 가리켜도 좋다.

**윤문안**

> **4-3**. [도전 문제] (앞부분 그대로) … 분류 정확도를 검증 지표로 사용하도록 [코드 4-3] 예제를 수정한 후,
> 검증 손실을 사용할 때와 최적 에포크 및 평가 정확도를 비교해 보자.